In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from natsort import natsorted

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
})


def plot_averaged_layers(
    chunks_to_plot,
    base_path,
    sample_name,
    plot_config=None,
    path_to_avg_data_csv=None,
):
    """
    Plot layer-wise averaged attention with full plotting control.
    
    Args:
        chunks_to_plot: List of chunk pairs to plot
        base_path: Base directory containing layer folders
        sample_name: Name of the sample file
        plot_config: Dictionary with plotting configuration
        path_to_avg_data_csv: Path to CSV file with normalization values (optional)
    """

    # ---------------- Config ----------------
    plot_config = plot_config or {}
    color_map = plot_config.get("colors", {})
    rename_map = plot_config.get("rename", {})
    highlight_segments = plot_config.get("highlight_segments", {})
    base_alpha = plot_config.get("base_alpha", 0.35)
    highlight_alpha = plot_config.get("highlight_alpha", 0.9)
    token_colors = plot_config.get("token_colors", {})
    
    # Line width configs - per-chunk or global defaults
    linewidth_map = plot_config.get("linewidths", {})
    default_linewidth = plot_config.get("linewidth", 2.0)
    highlight_lw_map = plot_config.get("highlight_linewidths", {})
    default_highlight_lw = plot_config.get("highlight_linewidth", 3.5)

    # Font configs
    font_config = plot_config.get("fonts", {})
    xlabel_fontsize = font_config.get("xlabel_fontsize", 12)
    ylabel_fontsize = font_config.get("ylabel_fontsize", 12)
    xlabel_fontweight = font_config.get("xlabel_fontweight", "normal")
    ylabel_fontweight = font_config.get("ylabel_fontweight", "normal")
    xlabel_fontfamily = font_config.get("xlabel_fontfamily", None)
    ylabel_fontfamily = font_config.get("ylabel_fontfamily", None)
    
    xtick_fontsize = font_config.get("xtick_fontsize", 10)
    ytick_fontsize = font_config.get("ytick_fontsize", 10)
    
    legend_cfg = plot_config.get("legend", {})
    show_spines = plot_config.get("show_spines", True)

    xlabel = plot_config.get("xlabel", "Step")
    ylabel = plot_config.get("ylabel", "Mean attention")

    # ---------------- Load normalization data ----------------
    normalization_dict = {}
    if path_to_avg_data_csv is not None:
        try:
            norm_df = pd.read_csv(path_to_avg_data_csv)
            normalization_dict = dict(zip(norm_df['Metric'], norm_df['Mean']))
            print(f"Loaded normalization data from {path_to_avg_data_csv}")
            print(f"Found {len(normalization_dict)} normalization values")
        except Exception as e:
            print(f"Warning: Could not load normalization CSV: {e}")
            normalization_dict = {}

    # ---------------- Chunk names ----------------
    chunk_names = [
        'im_start_0', 'user', 'blank', 'image', 'post_im_newline', 'text',
        'instruction', 'im_end', 'im_start_1', 'assistant',
        'post_assistant_newline', 'previously_generating_tokens',
        'currently_generating_token'
    ]

    attention_progression = {
        f"{tgt}__attends_to__{src}": []
        for i, src in enumerate(chunk_names)
        for tgt in chunk_names[i:]
    }

    # ---------------- Load data ----------------
    generated_tokens = None

    for layer_folder in natsorted(os.listdir(base_path)):
        if layer_folder == "logs":
            continue

        file_path = os.path.join(
            base_path, layer_folder, "attention_progression", sample_name
        )
        if not os.path.exists(file_path):
            continue

        data = np.load(file_path, allow_pickle=True)

        if generated_tokens is None:
            generated_tokens = list(data["generated_tokens"])
            generated_tokens[-1] = "<EOS>"

        for k in attention_progression:
            if k not in data:
                continue
            arr = np.array(data[k], dtype=object)
            arr = np.where(arr == None, np.nan, arr)
            attention_progression[k].append(arr.tolist())

    if generated_tokens is None:
        raise ValueError(f"No data found for {sample_name}")

    # ---------------- Mean ----------------
    attention_progression_avg = {
        k: np.nanmean(np.array(v, dtype=float), axis=0) if len(v) else np.array([])
        for k, v in attention_progression.items()
    }

    # ---------------- Apply normalization ----------------
    if normalization_dict:
        for chunk in chunks_to_plot:
            if chunk in attention_progression_avg and chunk in normalization_dict:
                norm_value = normalization_dict[chunk]
                attention_progression_avg[chunk] = attention_progression_avg[chunk] - norm_value
                print(f"Normalized {chunk} by subtracting {norm_value:.4f}")

    # ---------------- Plot ----------------
    num_tokens = len(generated_tokens)
    x = np.arange(num_tokens)

    # fig, ax = plt.subplots(figsize=(16, 9))
    fig, ax = plt.subplots(figsize=(6.5, 4))  # Single-column figure width

    for chunk in chunks_to_plot:
        y = attention_progression_avg.get(chunk)
        if y is None or not y.size:
            continue

        label = rename_map.get(chunk, chunk.replace("__attends_to__", " → "))
        color = color_map.get(chunk)
        
        # Get linewidth for this specific chunk, or use default
        lw = linewidth_map.get(chunk, default_linewidth)
        highlight_lw = highlight_lw_map.get(chunk, default_highlight_lw)

        line, = ax.plot(
            x,
            y,
            label=label,
            color=color,
            alpha=base_alpha,
            linewidth=lw,
            marker=".",
            zorder=5,
        )
        color = line.get_color()

        # ---- Highlight segments ----
        for start, end in highlight_segments.get(chunk, []):
            start = max(0, start)
            end = min(num_tokens - 1, end)
            ax.plot(
                x[start:end + 1],
                y[start:end + 1],
                color=color,
                linewidth=highlight_lw,
                alpha=highlight_alpha,
                zorder=10,
            )

    # ---------------- Bottom axis: step + shifted tokens ----------------
    bottom_tokens = [""] + generated_tokens[:-1]
    
    bottom_labels = []
    for i, tok in enumerate(bottom_tokens):
        display_tok = tok if tok else r'\n'
        bottom_labels.append(f"{i}\n{display_tok}")

    ax.set_xticks(x)
    ax.set_xticklabels(bottom_labels, rotation=45, ha="right", fontsize=xtick_fontsize)
    
    # Set xlabel with font properties
    xlabel_props = {
        'fontsize': xlabel_fontsize,
        'fontweight': xlabel_fontweight,
    }
    if xlabel_fontfamily:
        xlabel_props['fontfamily'] = xlabel_fontfamily
    ax.set_xlabel(xlabel, **xlabel_props)
    
    # Set ylabel with font properties
    ylabel_props = {
        'fontsize': ylabel_fontsize,
        'fontweight': ylabel_fontweight,
    }
    if ylabel_fontfamily:
        ylabel_props['fontfamily'] = ylabel_fontfamily
    
    # Update ylabel if normalization was applied
    if normalization_dict:
        ax.set_ylabel(ylabel + " (normalized)", **ylabel_props)
        ax.set_ylabel(ylabel, **ylabel_props)
    else:
        ax.set_ylabel(ylabel, **ylabel_props)

    # Set ytick font size
    ax.tick_params(axis='y', labelsize=ytick_fontsize)

    # ---------------- Top axis: tokens only ----------------
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(x)
    ax_top.set_xticklabels(generated_tokens, rotation=45, ha="left", fontsize=xtick_fontsize)

    # ---- Token coloring (both axes) ----
    for idx, tick in enumerate(ax.get_xticklabels()):
        if idx in token_colors:
            tick.set_color(token_colors[idx])
            tick.set_fontweight("bold")

    for idx, tick in enumerate(ax_top.get_xticklabels()):
        if idx in token_colors:
            tick.set_color(token_colors[idx])
            tick.set_fontweight("bold")

    # ---------------- Legend ----------------
    # ax.legend(
    #     frameon=False,
    #     ncol=legend_cfg.get("ncols", 1),
    #     fontsize=legend_cfg.get("fontsize", 11),
    #     loc=legend_cfg.get("loc", "best"),
    #     bbox_to_anchor=legend_cfg.get("bbox_to_anchor", None),
    #     prop={
    #         'size': legend_cfg.get("fontsize", 11),
    #         'weight': legend_cfg.get("fontweight", "normal"),
    #         'family': legend_cfg.get("fontfamily", None),
    #     }
    # )

    from matplotlib.lines import Line2D

    legend_handles = []

    for chunk in chunks_to_plot:
        y = attention_progression_avg.get(chunk)
        if y is None or not y.size:
            continue

        label = rename_map.get(chunk, chunk.replace("__attends_to__", " → "))
        color = color_map.get(chunk)
        lw = linewidth_map.get(chunk, default_linewidth)

        legend_handles.append(
            Line2D(
                [0], [0],
                color=color,
                linewidth=lw * 4,
                alpha=1.0,          # <-- FORCE FULL OPACITY
                label=label
            )
        )

    ax.legend(
        handles=legend_handles,
        frameon=False,
        ncol=legend_cfg.get("ncols", 1),
        fontsize=legend_cfg.get("fontsize", 11),
        loc=legend_cfg.get("loc", "best"),
        bbox_to_anchor=legend_cfg.get("bbox_to_anchor", None),
        prop={
            'size': legend_cfg.get("fontsize", 11),
            'weight': legend_cfg.get("fontweight", "normal"),
            'family': legend_cfg.get("fontfamily", None),
        }
    )


    ax.grid(alpha=0.2)

    # ---------------- Spines ----------------
    if not show_spines:
        for spine in ax.spines.values():
            spine.set_visible(False)
        for spine in ax_top.spines.values():
            spine.set_visible(False)
    
    plt.tight_layout()
    # plt.show()
    # plt.close()

    # return attention_progression_avg
    return fig, ax, attention_progression_avg

In [ ]:
plot_config = {
    # ============ Colors ============
    "colors": {
        "currently_generating_token__attends_to__image": "#FFA500",
        "currently_generating_token__attends_to__text": "#267E59",
        "currently_generating_token__attends_to__previously_generating_tokens": "#C01BA7",
        "currently_generating_token__attends_to__instruction": "#D25B5B",
        "currently_generating_token__attends_to__im_start_0": "#2A4B89",
    },
    
    # ============ Line Widths ============
    # Global defaults for all lines
    "linewidth": 0.5,
    "highlight_linewidth": 2.0,
    
    # Per-chunk line widths (optional - overrides global default)
    "linewidths": {
        # Example: make specific lines thicker/thinner
        # "currently_generating_token__attends_to__previously_generating_tokens": 0.1,
        # "currently_generating_token__attends_to__im_start_0": 0.1
    },
    
    # Per-chunk highlight widths (optional - overrides global highlight default)
    "highlight_linewidths": {
        # Example: make specific highlighted segments thicker
        # "currently_generating_token__attends_to__image": 2.0,
    },
    
    # ============ Labels & Renaming ============
    "xlabel": "Input Token",
    "ylabel": "Normalized Attention Scores",
    # "ylabel": "Raw Attention Scores",
    
    "rename": {
        "currently_generating_token__attends_to__image": "CGT → Image",
        "currently_generating_token__attends_to__text": "CGT → Text",
        "currently_generating_token__attends_to__instruction": "CGT → Instruction",
        "currently_generating_token__attends_to__previously_generating_tokens": "CGT → Previous",
        "currently_generating_token__attends_to__im_start_0": "CGT → im_start_0",

    },
    
    # ============ Highlight Segments ============
    "highlight_segments": {
        "currently_generating_token__attends_to__image": [(5, 8)],
        "currently_generating_token__attends_to__instruction": [(6, 8)],
        "currently_generating_token__attends_to__text": [(14, 18)],
        # "currently_generating_token__attends_to__im_start_0": [(0, 22)],
        # "currently_generating_token__attends_to__previously_generating_tokens": [(0, 22)],
    },
    
    # ============ Alpha/Transparency ============
    "base_alpha": 0.3,
    "highlight_alpha": 0.95,
    
    # ============ Token Colors ============
    "token_colors": {
        6: "#FFA500",
        # 7: "#FFA500",
        16: "#267E59",
        17: "#267E59",
        18: "#267E59",
        19: "#267E59",
        20: "#267E59",
        21: "#267E59",
    },
    
    # ============ Spines ============
    "show_spines": False,
    
    # ============ Font Configuration ============
    "fonts": {
        # X-axis label
        "xlabel_fontsize": 8,
        # "xlabel_fontweight": "normal",
        # "xlabel_fontfamily": "serif",
        
        # Y-axis label
        "ylabel_fontsize": 8,
        # "ylabel_fontweight": "normal",
        # "ylabel_fontfamily": "serif",
        
        # Tick labels
        "xtick_fontsize": 7,
        "ytick_fontsize": 7,
    },
    
    # ============ Legend ============
    "legend": {
        "ncols": 3,
        "fontsize": 8,
        # "fontweight": "normal",
        # "fontfamily": "sans-serif",
        "loc": "best",
        # "bbox_to_anchor": None,  # e.g., (1.05, 1) for outside plot
    },
}



In [ ]:
fig, _, _ = plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/qwen35vl_demo/fruit_math_vanilla",
    sample_name="fruit_math__image-actual__fruit_math_prompt__fruit-Watermelon__math-330000__id-558.npz",
    plot_config=plot_config,
    # path_to_avg_data_csv="<REPO_ROOT>/data/global_means/FrMaSc/FrMaSc__lov_7b.csv",
)

#save hires fig
# fig.savefig("<REPO_ROOT>/data/plots/figure_1/lineplot_Norm_lov7b_id_235_frmasp.pdf", dpi=300)

In [ ]:
fig, _, _ = plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/otat/FrMaSc/lov_7b",
    sample_name="fruit_math__image-actual__fruit_math_prompt__fruit-Strawberry__math-12__id-235.npz",
    plot_config=plot_config,
    path_to_avg_data_csv="<REPO_ROOT>/data/global_means/FrMaSc/FrMaSc__lov_7b.csv",
)

#save hires fig
fig.savefig("<REPO_ROOT>/data/plots/figure_1/lineplot_Norm_lov7b_id_235_frmasp.pdf", dpi=300)

# Supplementary

In [ ]:
plot_config = {
    # ============ Colors ============
    "colors": {
        "currently_generating_token__attends_to__image": "#FFA500",
        "currently_generating_token__attends_to__text": "#267E59",
        "currently_generating_token__attends_to__previously_generating_tokens": "#C01BA7",
        "currently_generating_token__attends_to__instruction": "#D25B5B",
        "currently_generating_token__attends_to__im_start_0": "#2A4B89",
    },
    
    # ============ Line Widths ============
    # Global defaults for all lines
    "linewidth": 0.5,
    "highlight_linewidth": 2.0,
    
    # Per-chunk line widths (optional - overrides global default)
    "linewidths": {
        # Example: make specific lines thicker/thinner
        # "currently_generating_token__attends_to__previously_generating_tokens": 0.1,
        # "currently_generating_token__attends_to__im_start_0": 0.1
    },
    
    # Per-chunk highlight widths (optional - overrides global highlight default)
    "highlight_linewidths": {
        # Example: make specific highlighted segments thicker
        # "currently_generating_token__attends_to__image": 2.0,
    },
    
    # ============ Labels & Renaming ============
    "xlabel": "Input Token",
    "ylabel": "Normalized Attention Scores",
    # "ylabel": "Raw Attention Scores",
    
    "rename": {
        "currently_generating_token__attends_to__image": "CGT → Image",
        "currently_generating_token__attends_to__text": "CGT → Text",
        "currently_generating_token__attends_to__instruction": "CGT → Instruction",
        "currently_generating_token__attends_to__previously_generating_tokens": "CGT → Previous",
        "currently_generating_token__attends_to__im_start_0": "CGT → im_start_0",

    },
    
    # ============ Highlight Segments ============
    "highlight_segments": {
        "currently_generating_token__attends_to__image": [(5, 8)],
        "currently_generating_token__attends_to__instruction": [(8, 9)],
        "currently_generating_token__attends_to__text": [(16, 17)],
        # "currently_generating_token__attends_to__im_start_0": [(0, 22)],
        # "currently_generating_token__attends_to__previously_generating_tokens": [(0, 22)],
    },
    
    # ============ Alpha/Transparency ============
    "base_alpha": 0.3,
    "highlight_alpha": 0.95,
    
    # ============ Token Colors ============
    "token_colors": {
        7: "#FFA500",
        # 7: "#FFA500",
        # 7: "#FFA500",
        17: "#267E59",
        # 10: "#267E59",
        # 11: "#267E59",
        # 12: "#267E59",
        # 13: "#267E59",
    },
    
    # ============ Spines ============
    "show_spines": False,
    
    # ============ Font Configuration ============
    "fonts": {
        # X-axis label
        "xlabel_fontsize": 8,
        # "xlabel_fontweight": "normal",
        # "xlabel_fontfamily": "serif",
        
        # Y-axis label
        "ylabel_fontsize": 8,
        # "ylabel_fontweight": "normal",
        # "ylabel_fontfamily": "serif",
        
        # Tick labels
        "xtick_fontsize": 7,
        "ytick_fontsize": 7,
    },
    
    # ============ Legend ============
    "legend": {
        "ncols": 3,
        "fontsize": 8,
        # "fontweight": "normal",
        # "fontfamily": "sans-serif",
        "loc": "best",
        # "bbox_to_anchor": None,  # e.g., (1.05, 1) for outside plot
    },
}



In [ ]:
fig, _, _ = plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/otat/FrMaSc/qvl_7b",
    sample_name="fruit_math__image-actual__fruit_math_prompt__fruit-Apple__math-4__id-123.npz",
    plot_config=plot_config,
    path_to_avg_data_csv="<REPO_ROOT>/data/global_means/FrMaSc/FrMaSc__qvl_7b.csv",
)

#save hires fig
fig.savefig("<REPO_ROOT>/data/plots/Supplementary/line_plots_frmasc/qvl_7b_id123.pdf", dpi=300)